In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

dbutils.widgets.text("catalogo", "proyecto_ecommerce")
catalogo = dbutils.widgets.get("catalogo")

df_clientes = spark.table(f"{catalogo}.silver.clientes")
df_productos = spark.table(f"{catalogo}.silver.productos")
df_ordenes = spark.table(f"{catalogo}.silver.ordenes")

In [0]:
fechas_distintas = df_ordenes.select("fecha").distinct()

MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Setiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}
mes_udf = F.udf(lambda m: MESES_ES[m])

DIAS_ES = {
    "Monday": "Lunes", "Tuesday": "Martes", "Wednesday": "Miercoles",
    "Thursday": "Jueves", "Friday": "Viernes", "Saturday": "Sabado", "Sunday": "Domingo",
}
dia_udf = F.udf(lambda d: DIAS_ES[d])

dim_tiempo = (
    fechas_distintas
    .withColumn("fecha_id", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("nombre_mes", mes_udf(F.col("mes")))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("dia_semana", dia_udf(F.date_format("fecha", "EEEE")))
    .select("fecha_id", "fecha", "anio", "mes", "nombre_mes", "trimestre", "dia_semana")
)

display(dim_tiempo.orderBy("fecha").limit(10))
print(f"gold.dim_tiempo (aún sin guardar) -> {dim_tiempo.count()} filas")

In [0]:
(dim_tiempo.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.dim_tiempo"))

print(f"Guardado: {catalogo}.gold.dim_tiempo -> {dim_tiempo.count()} filas")